In [ ]:
!apt-get install -y python3-rdkit
!pip install rdkit

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fonts-freefont-ttf libboost-iostreams1.74.0 libboost-python1.74.0
  libboost-serialization1.74.0 libcoordgen3 libmaeparser1 librdkit1
  python3-numpy rdkit-data
Suggested packages:
  python-numpy-doc python3-dev python3-pytest rdkit-doc
The following NEW packages will be installed:
  fonts-freefont-ttf libboost-iostreams1.74.0 libboost-python1.74.0
  libboost-serialization1.74.0 libcoordgen3 libmaeparser1 librdkit1
  python3-numpy python3-rdkit rdkit-data
0 upgraded, 10 newly installed, 0 to remove and 5 not upgraded.
Need to get 28.0 MB of archives.
After this operation, 136 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 fonts-freefont-ttf all 20120503-10build1 [2,388 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libboost-iostreams1.74.0 amd64 1.74.0-14ubuntu3 [245 kB]
Ge

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving drug_taste_dataset.csv to drug_taste_dataset (2).csv


In [ ]:
import pandas as pd

# Read file WITHOUT assuming separator
data = pd.read_csv(list(uploaded.keys())[0])

# If everything is in ONE column → fix it
if len(data.columns) == 1:
    data = data[data.columns[0]].str.split("\t", expand=True)

# Rename columns properly
data.columns = ["SMILES", "Taste"]

# Clean values
data["Taste"] = data["Taste"].astype(str).str.strip()

print("Columns:", data.columns)
print("Rows:", len(data))
print(data.head())

Columns: Index(['SMILES', 'Taste'], dtype='object')
Rows: 32
                          SMILES   Taste
0          CC(=O)NC1=CC=C(O)C=C1  Bitter
1       CC(=O)OC1=CC=CC=C1C(=O)O  Bitter
2  CC(C)CC1=CC=C(C=C1)C(C)C(=O)O  Bitter
3              CN(C)C(=N)NC(=N)N  Bitter
4   CN1C=NC2=C1C(=O)N(C(=O)N2C)C  Bitter


In [ ]:
print(data["Taste"].unique())

['Bitter' 'nan' 'Sweet']


In [ ]:
# Remove 'nan' string rows
data = data[data["Taste"] != "nan"]

# Map values
data["Taste"] = data["Taste"].map({
    "Sweet": 1,
    "Bitter": 0
})

print("Unique after cleaning:", data["Taste"].unique())
print("Rows:", len(data))

Unique after cleaning: [0 1]
Rows: 30


In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np

def smiles_to_fp(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    return np.array(AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024))

In [ ]:
X = []
y = []

for _, row in data.iterrows():
    fp = smiles_to_fp(row["SMILES"])
    if fp is not None:
        X.append(fp)
        y.append(row["Taste"])

X = np.array(X)
y = np.array(y)

print(X.shape, y.shape)

(29, 1024) (29,)


[10:36:19] DEPRECATION WARNING: please use MorganGenerator
[10:36:19] DEPRECATION WARNING: please use MorganGenerator
[10:36:19] DEPRECATION WARNING: please use MorganGenerator
[10:36:19] DEPRECATION WARNING: please use MorganGenerator
[10:36:19] DEPRECATION WARNING: please use MorganGenerator
[10:36:19] DEPRECATION WARNING: please use MorganGenerator
[10:36:19] DEPRECATION WARNING: please use MorganGenerator
[10:36:19] DEPRECATION WARNING: please use MorganGenerator
[10:36:19] SMILES Parse Error: unclosed ring for input: 'CN1CCC23C4C1CC5=C2C=CC(O)=C5O3'
[10:36:19] DEPRECATION WARNING: please use MorganGenerator
[10:36:19] DEPRECATION WARNING: please use MorganGenerator
[10:36:19] DEPRECATION WARNING: please use MorganGenerator
[10:36:19] DEPRECATION WARNING: please use MorganGenerator
[10:36:19] DEPRECATION WARNING: please use MorganGenerator
[10:36:19] DEPRECATION WARNING: please use MorganGenerator
[10:36:19] DEPRECATION WARNING: please use MorganGenerator
[10:36:19] DEPRECATION WAR

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=1000)
model.fit(X_train, y_train)

print("Trained ✅")

Trained ✅


In [ ]:
import joblib
joblib.dump(model, "taste_model.pkl")
print("Saved ✅")

Saved ✅


In [ ]:
from google.colab import files
files.download("taste_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>